In [10]:
import json
import os
from pathlib import Path

import numpy as np
import torch

import pandas as pd


In [2]:
base = Path("../attributions")


In [23]:
def temporal_saliency(dataset, model, window, subject=None, method="saliency", targets=[0, 1]):
    files = os.listdir(base)
    files_to_load = []
    for file in files:
        if window in file and dataset in file and model in file:
            if subject is not None:
                if f"_{subject}.pt" in file:
                    files_to_load.append(file)
            else:
                files_to_load.append(file)

    saliency_list = []

    for file in files_to_load:
        data = torch.load(base / file, weights_only=False)
        for target in targets:
            saliency = data[target][method]
            saliency_list.append(saliency)

    saliency_all = np.array(saliency_list)

    saliency_temporal = saliency_all.mean(1)

    saliency_temporal = saliency_temporal / saliency_temporal.sum(1)[:, None] * 100

    t1 = window.split("_")[1]
    t2 = window.split("_")[2]

    t1 = float(f"{t1[0]}.{t1[1]}")
    t2 = float(f"{t2[0]}.{t2[1]}")

    times = np.linspace(t1, t2, saliency_temporal.shape[1])

    saliency_temporal = saliency_temporal.mean(0)

    df_dict = {"time": times, "attribution": saliency_temporal}
    df = pd.DataFrame(df_dict)

    return df


def spatial_saliency(dataset, model, window, subject=None, method="saliency", targets=[0, 1],
                     montage="standard_1005"):
    with open(f"../Dataset/{dataset}/meta_data.json", "r") as f:
        info = json.load(f)
    ch_names = info["ch_names"]

    files = os.listdir(base)
    files_to_load = []
    for file in files:
        if window in file and dataset in file and model in file:
            if subject is not None:
                if f"_{subject}.pt" in file:
                    files_to_load.append(file)
            else:
                files_to_load.append(file)

    saliency_list = []

    for file in files_to_load:
        data = torch.load(base / file, weights_only=False)
        for target in targets:
            saliency = data[target][method]
            saliency_list.append(saliency)

    saliency_all = np.array(saliency_list)

    saliency_spatial = saliency_all.mean(2)
    saliency_spatial = saliency_spatial / saliency_spatial.sum(1)[:, None] * 100

    saliency_spatial = saliency_spatial.mean(0)

    df_dict = {"channel": [], "attribution": []}
    for i, ch in enumerate(ch_names):
        df_dict["channel"].append(ch)
        df_dict["attribution"].append(saliency_spatial[i])

    df = pd.DataFrame(df_dict)

    return df


df = temporal_saliency("Dreyer2023", model="Deep4Net", window="w_00_40", method="saliency")
print(df[df["attribution"] == df["attribution"].max()])

df = temporal_saliency("Dreyer2023", model="Deep4Net", window="w_05_45", method="saliency")
print(df[df["attribution"] == df["attribution"].max()])




        time  attribution
47  0.188188     0.624139
         time  attribution
228  1.412913     0.189153


In [ ]:
df = spatial_saliency("Dreyer2023", model="Deep4Net", window="w_00_40", method="saliency")
print(df.to_string())

df = spatial_saliency("Dreyer2023", model="Deep4Net", window="w_05_45", method="saliency")
print(df.to_string())
